<a href="https://colab.research.google.com/github/bsheese/cs377/blob/18_classification_redesign/18_classification/18_1_Classification_Basics/18_1_0_Classification_Foundations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classification Foundations: Credit Default Prediction

**Author:** Brad Sheese

---

## Why Not Linear Regression for Classification?

Linear regression outputs unbounded numbers. But probabilities must be between 0 and 1.

If we try to use linear regression for classification, the line will extend beyond 0-1, suggesting negative probabilities or >100% chances - which makes no sense!

**Solution**: Use the **sigmoid function** to squash all outputs between 0 and 1.

### The Sigmoid Function

$$P(Y=1) = \frac{1}{1 + e^{-(\beta_0 + \beta_1X_1 + ...)}}$$

This 'S-curve' naturally bounds predictions between 0 and 1.

### Learning Objectives
By the end of this notebook, you will:
1. Understand why classification needs special methods
2. Load and explore an imbalanced dataset
3. Train a Logistic Regression classifier
4. Interpret probability outputs and the decision threshold

## Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Load the Credit Card Default dataset from OpenML
from sklearn.datasets import fetch_openml

print('Loading dataset...')
data = fetch_openml(name='credit-g', version=1, as_frame=True)
df = data.frame

print(f'Dataset shape: {df.shape}')
print(f'\nFirst few rows:')
df.head()

## Understanding Class Imbalance

**Key insight**: Our dataset has ~78% 'good' credits and ~22% 'bad' defaults.

This imbalance is CRITICAL because:
- A model that predicts 'good' for everyone gets 78% accuracy!
- But it's completely useless at detecting defaults
- This is called the 'Accuracy Paradox'

In [ ]:
# Check target variable distribution
print('Target variable distribution:')
class_counts = df['class'].value_counts()
print(class_counts)

bad_pct = (df['class'] == 'bad').mean() * 100
good_pct = (df['class'] == 'good').mean() * 100
print(f'\nPercentage breakdown:')
print(f'  Good: {good_pct:.1f}%')
print(f'  Bad:  {bad_pct:.1f}%')

# Visualize imbalance
plt.figure(figsize=(8, 5))
colors = ['#2ecc71', '#e74c3c']  # green for good, red for bad
plt.bar(['Good', 'Bad'], [good_pct, bad_pct], color=colors, edgecolor='black')
plt.ylabel('Percentage (%)')
plt.title('Class Imbalance in Credit Default Dataset')
for i, (label, pct) in enumerate(zip(['Good', 'Bad'], [good_pct, bad_pct])):
    plt.text(i, pct + 1, f'{pct:.1f}%', ha='center', fontweight='bold')
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

print(f'\nNaive baseline (predict all Good): {good_pct:.1f}% accuracy')

## Data Preprocessing

We need to:
1. Convert target to binary (1 = default, 0 = good)
2. One-hot encode categorical features
3. Split into train/test sets

In [ ]:
# Convert target to binary: good = 0, bad = 1
y = (df['class'] == 'bad').astype(int)
X = df.drop(columns=['class'])

# Identify column types
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(include=['number']).columns.tolist()

print(f'Categorical columns: {len(cat_cols)}')
print(f'Numerical columns: {len(num_cols)}')

# One-hot encode
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)
print(f'\nFeatures after encoding: {X_encoded.shape[1]}')

# Train-test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print(f'\nTraining set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')
print(f'Test set default rate: {y_test.mean()*100:.1f}%')

## Training Logistic Regression

Logistic Regression uses the sigmoid to output probabilities between 0 and 1.

The model learns coefficients that maximize the probability of correct classification.

In [ ]:
# Scale features (important for convergence)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Logistic Regression
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

# Get predictions
y_pred = model.predict(X_test_scaled)       # Hard predictions (0 or 1)
y_proba = model.predict_proba(X_test_scaled)[:, 1]  # Soft predictions (probability of default)

print('Model trained successfully!')
print(f'\nFirst 10 predictions:')
print(f'{'Actual':>8} {'Predicted':>10} {'Probability':>12}')
for i in range(10):
    print(f'{y_test.iloc[i]:>8} {y_pred[i]:>10} {y_proba[i]:>12.3f}')

## Understanding the Decision Threshold

**Default threshold = 0.5**
- If P(default) >= 0.5, predict 'default' (1)
- If P(default) < 0.5, predict 'good' (0)

This threshold is ARBITRARY - we'll explore tuning it in Notebook 3.

In [ ]:
# Visualize probability distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Histogram of all probabilities
axes[0].hist(y_proba, bins=40, edgecolor='black', alpha=0.7)
axes[0].axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Threshold=0.5')
axes[0].set_xlabel('Predicted Probability of Default')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Predicted Probabilities')
axes[0].legend()

# Right: Overlapping histograms by actual class
axes[1].hist(y_proba[y_test==0], bins=30, alpha=0.7, label='Actual: Good (0)', color='green')
axes[1].hist(y_proba[y_test==1], bins=30, alpha=0.7, label='Actual: Bad (1)', color='red')
axes[1].axvline(x=0.5, color='black', linestyle='--', linewidth=2, label='Threshold=0.5')
axes[1].set_xlabel('Predicted Probability of Default')
axes[1].set_ylabel('Count')
axes[1].set_title('Probabilities by Actual Class')
axes[1].legend()

plt.tight_layout()
plt.show()

# Calculate accuracy
accuracy = (y_pred == y_test).mean()
baseline = (y_test == 0).mean()
print(f'\nModel Accuracy: {accuracy*100:.2f}%')
print(f'Baseline (predict all 0): {baseline*100:.2f}%')
print(f'Improvement over baseline: +{(accuracy - baseline)*100:.2f}%')

## Key Takeaways

1. **Classification needs sigmoid** to bound probabilities between 0-1
2. **Class imbalance is critical** - always check distribution!
3. **Soft predictions** are probabilities, **hard predictions** are binary
4. **Default threshold = 0.5** but this is arbitrary
5. **Accuracy alone is misleading** on imbalanced data

In Notebook 2, we'll learn how to properly evaluate classifiers using the confusion matrix and other metrics.